# Wavenet

In [1]:
import numpy as np
import pywt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import datasets, transforms
from sktime.datasets import load_from_tsfile_to_dataframe
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.nn.functional as F
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import json
import os


In [2]:
def get_resnext_binary_model():
    # 1. Load pre-trained ResNeXt-101 (32x8d is the standard high-performance variant)
    # Note: resnext101_32x8d is generally preferred over 64x4d for most tasks
    model = models.resnext101_32x8d(weights=models.ResNeXt101_32X8D_Weights.DEFAULT)
    
    # --- Step 1: Handle 1-Channel Input ---
    # Convert the first convolution from 3-channel (RGB) to 1-channel (CWT)
    old_conv = model.conv1
    model.conv1 = nn.Conv2d(1, old_conv.out_channels, 
                            kernel_size=old_conv.kernel_size, 
                            stride=old_conv.stride, 
                            padding=old_conv.padding, 
                            bias=False)
    
    # Average the RGB weights to initialize the single-channel weights
    with torch.no_grad():
        model.conv1.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)

    # --- Step 2: Freeze the Backbone ---
    for param in model.parameters():
        param.requires_grad = False

    # --- Step 3: Replace the Head ---
    # The feature count (num_ftrs) for ResNeXt-101 is the same as ResNeXt-50 (2048)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    
    # Enable gradients for the head and the modified first layer
    for param in model.fc.parameters():
        param.requires_grad = True
        
    # Optional: If you want to train the input layer too (since we modified it)
    for param in model.conv1.parameters():
        param.requires_grad = True
        
    return model

In [3]:
def model_fit_resnext(train_loader, device):
    """
    Fits the ResNeXt model by training only the unfrozen layers 
    (the new 1-channel conv1 and the classification head).
    """
    # 1. Initialize the model using the helper function we created
    model = get_resnext_binary_model().to(device)
    
    # 2. Binary Classification setup
    criterion = nn.BCELoss()

    # IMPORTANT: Only pass parameters that require gradients to the optimizer.
    # This excludes the frozen ResNeXt backbone.
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), 
        lr=1e-4, 
        weight_decay=1e-4
    )
    
    # Keeping your existing scheduler logic
    scheduler = StepLR(optimizer, step_size=8, gamma=0.3)
    
    epochs = 50 # ResNeXt usually converges faster than scratch models; adjusted down from 150
    
    for epoch in tqdm(range(epochs), desc="Training ResNeXt:"):
        model.train()
        total_loss = 0
    
        for x, y in train_loader:
            x = x.to(device)
            # Binary targets must be float and [batch_size, 1] for BCELoss
            y = y.to(device).float().unsqueeze(1) 
    
            outputs = model(x)
            loss = criterion(outputs, y)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()
    
        scheduler.step()

    return model

In [4]:
def evaluate_metrics(model, dataloader, device, threshold=0.5):
    """
    Evaluates binary classification metrics for the CWT-based model.
    """
    model.eval()

    y_true_all = []
    y_pred_all = []
    y_proba_all = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            # Binary targets for BCELoss/Scikit-learn
            y_true = y.to(device).float().view(-1)
            
            outputs = model(x)

            # Extract probabilities (since the model ends with Sigmoid)
            probs = outputs.view(-1).float()
            preds = (probs >= threshold).float()

            y_proba_all.append(probs.cpu())
            y_true_all.append(y_true.cpu())
            y_pred_all.append(preds.cpu())

    # Concatenate all batches
    y_test = torch.cat(y_true_all).numpy()
    y_pred = torch.cat(y_pred_all).numpy()
    y_proba = torch.cat(y_proba_all).numpy()

    # Calculate Standard Binary Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        # Prevents crashing if the batch only contains one class
        auc = float("nan")

    return accuracy, recall, precision, f1, auc

In [5]:
def cwt_transform(signal, wavelet='morl', scales=np.arange(1, 21)):
    signal = np.array(signal)
    coefs, _ = pywt.cwt(signal, scales, wavelet)
    return coefs  # shape: [scale, time]

In [6]:
class CWTTimeSeriesDataset(Dataset):
    def __init__(self, x_df, y_list, wavelet, scales):
        self.samples = []
        self.labels = []

        for i in tqdm(range(len(x_df)), desc="Processing CWT"):
            ts = x_df.iloc[i].iloc[0]
            img = cwt_transform(ts, wavelet, scales)
            img = np.expand_dims(img, axis=0)  # shape: [1, scale, time]
            self.samples.append(torch.tensor(img, dtype=torch.float32))
            self.labels.append(int(y_list[i]))

    def plot(self, idx):
        img = self.samples[idx].squeeze(0).numpy()  # shape: [scale, time]
        plt.imshow(
            img,
            extent=[0, img.shape[1], 1, img.shape[0] + 1],
            cmap='PRGn',
            aspect='auto',
            vmax=abs(img).max(),
            vmin=-abs(img).max()
        )
        plt.title(f"CWT of Sample {idx}")
        plt.ylabel('Scale')
        plt.xlabel('Time')
        plt.colorbar(label='Amplitude')
        plt.show()

    def resize_images(self, new_size=(224, 224)):
        resized_samples = []
        for img in tqdm(self.samples, desc="Resizing images"):
            resized_img = F.interpolate(img.unsqueeze(0), size=new_size, mode='bilinear', align_corners=False)
            resized_samples.append(resized_img.squeeze(0))
        self.samples = resized_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]


In [7]:
def wl_train_and_test(data_name):
    wavelet = 'morl'
    scales = np.arange(1, 21) # 20 scales (height)
    
    # --- 1. Load Data ---
    data_id = data_name[4:6]
    file_path_train = f"data/{data_name}/TokenItaly_vers0/TokenItaly_vers0_TRAIN.ts"
    file_path_test = f"data/{data_name}/TokenItaly_vers0/TokenItaly_vers0_TEST.ts"

    x_train, y_train = load_from_tsfile_to_dataframe(file_path_train)
    x_test, y_test = load_from_tsfile_to_dataframe(file_path_test)

    # --- 2. Create and Resize Datasets ---
    train_dataset = CWTTimeSeriesDataset(x_train, y_train, wavelet, scales)
    val_dataset = CWTTimeSeriesDataset(x_test, y_test, wavelet, scales)
    
    # ResNeXt requires 224x224 to leverage pre-trained spatial features
    print(f"Resizing images for {data_name}...")
    train_dataset.resize_images(new_size=(224, 224))
    val_dataset.resize_images(new_size=(224, 224))
    
    # Lower batch size (16) is usually safer for ResNeXt on consumer GPUs
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Using device: {device} for ResNeXt training on {data_name}')

    # --- 3. Train Model ---
    # We only return the model now, as requested
    model = model_fit_resnext(train_loader, device)
    
    # --- 4. Evaluate ---
    accuracy, recall, precision, f1, auc = evaluate_metrics(model, val_loader, device)
    
    result_dict = {
        "data_id" : data_id,
        "data_name" : data_name,
        "model" : "ResNeXt50_TransferLearning",
        "Accuracy" : accuracy,
        "Precision" : precision,
        "Recall" : recall,
        "F1-score" : f1,
        "ROC-AUC score" : auc
    }

    return result_dict

In [ ]:
json_path = 'results.json'
csv_path = 'results.csv'

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        all_results = json.load(f)
else:
    all_results = []

data_root_path = 'data'

# Define target datasets to run (can be modified)
target_ids = ['10', '11', '22']

all_data_names = sorted([
    name for name in os.listdir(data_root_path)
    if os.path.isdir(os.path.join(data_root_path, name)) and name.startswith('data')
])

# Filter datasets to only include those in target_ids
data_names = []
for name in all_data_names:
    # Check if name matches data{id} or data{id}_...
    for tid in target_ids:
        if name == f"data{tid}" or name.startswith(f"data{tid}_"):
            data_names.append(name)
            break




c = 0
for data_name in data_names:
    result_dict = wl_train_and_test(data_name)
    all_results.append(result_dict)
    c += 1
    print(data_name)

with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)


df = pd.DataFrame(all_results)
df.to_csv(csv_path, index=False)

print(f"{c} data is trained and results added to the files...")

Processing CWT: 100%|██████████████████████████████████████████████████████████████████████████████| 4970/4970 [00:01<00:00, 3091.83it/s]


Resizing images for data10_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_full_data...


Resizing images: 100%|█████████████████████████████████████████████████████████████████████████████| 4970/4970 [00:00<00:00, 9363.45it/s]


Using device: cuda for ResNeXt training on data10_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_1_ratio_full_data


Downloading: "https://download.pytorch.org/models/resnext101_32x8d-110c445d.pth" to /root/.cache/torch/hub/checkpoints/resnext101_32x8d-110c445d.pth
100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 340M/340M [00:16<00:00, 22.1MB/s]
Training ResNeXt:: 100%|█████████████████████████████████████████████████████████████████████████████████| 50/50 [53:53<00:00, 64.68s/it]


...

Processing CWT: 100%|████████████████████████████████████████████████████████████████████████████| 12425/12425 [00:04<00:00, 3032.71it/s]


Resizing images for data11_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_4_ratio_full_data...


Resizing images: 100%|███████████████████████████████████████████████████████████████████████████| 12425/12425 [00:01<00:00, 8122.80it/s]


Using device: cuda for ResNeXt training on data11_IC1_FM_label_wl_20_pw_0_unwo_dropped_1_4_ratio_full_data


Training ResNeXt::   0%|                                                                                          | 0/50 [00:00<?, ?it/s]